<a href="https://colab.research.google.com/github/nbanik-dev/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring (with a ranking output).**

The editor doesn't need a hard yes/no per page — they need a queue: 30,000 pages sorted
so the top ~50 are the ones most worth reviewing this cycle. That's a scoring problem —
every page gets a continuous risk score (probability of being a genuine decline
opportunity), and the queue is that score sorted descending. It's built on top of a binary
classifier (declining vs not), but the deliverable an editor actually uses is the ranked
list, not the raw label — so I'm framing the task as scoring/ranking, not plain
classification.

In [ ]:
# Framing only — no computation needed for this section.
print("Task type: scoring -> produces a ranked review queue for editors.")

Task type: scoring -> produces a ranked review queue for editors.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy): `is_declining_label`** — a page is labeled "declining" based on its
observed `trend_direction` (derived from comparing `impressions_last_30d` vs
`impressions_prev_30d`). It's an **observed outcome**, not a hand-defined rule — the label
comes from real traffic history, not from me deciding what "risk" looks like.

This is a proxy, not a perfect target: "trending down in impressions" isn't the same as
"actually worth an editor's time to review." A page could decline for reasons outside an
editor's control (seasonality, a SERP feature change) that no refresh would fix. I'll flag
that gap explicitly rather than treat the label as ground truth.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nbanik-dev/flyrank_ml_internship"
REPO_DIR = "flyrank_ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=False))
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Working dir: /content/flyrank_ml_internship/flyrank_ml_internship
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.** Since the editor only reviews a fixed, small number of pages per
cycle (~50), what matters is not overall accuracy across all 30,000 pages — it's *how many
of the top 50 the model surfaces are genuinely worth reviewing*. A model with mediocre
overall accuracy but a sharp, reliable top-50 is more useful here than one with high
overall accuracy but a noisy top.

Reference numbers from the pipeline I already ran (`scripts/run_all.py`, see
`outputs/model_report.md`):

| Model | Precision@50 |
|---|---:|
| baseline_rules (hand-written) | 0.240 |
| logistic_regression | 0.400 |
| decision_tree | 0.540 |
| random_forest (best) | 0.740 |

"Good" for my capstone means beating the 0.24 hand-written baseline by a clear margin —
the pipeline's own random_forest result (0.74) sets a realistic ceiling to aim toward,
not necessarily to match.

In [ ]:
# Reference metrics already computed by the reference pipeline (scripts/run_all.py).
metrics = {
    "baseline_rules": 0.240,
    "logistic_regression": 0.400,
    "decision_tree": 0.540,
    "random_forest": 0.740,
}
for name, p50 in metrics.items():
    print(f"{name:20s} precision@50 = {p50:.3f}")

baseline_rules       precision@50 = 0.240
logistic_regression  precision@50 = 0.400
decision_tree        precision@50 = 0.540
random_forest        precision@50 = 0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), aggregated over its trailing 90-day search
and engagement history. Not a client, not a day, not a keyword — a single page, scored
once per refresh cycle.

In [ ]:
cols = [
    "content_id", "content_type", "word_count", "avg_position", "ctr",
    "impressions_90d", "trend_direction", "is_declining_label",
]
print(f"Unit of analysis: one row = one content page. Total pages: {len(df)}")
df[cols].head(5)

Unit of analysis: one row = one content page. Total pages: 30000


,content_id,content_type,word_count,avg_position,ctr,impressions_90d,trend_direction,is_declining_label
0,content_304f48230142,keyword article,3221.0,10.6,0.76,3803,down,1
1,content_a1fb4e703a9e,keyword article,2481.0,20.3,0.05,15320,down,1
2,content_9aa793d4d895,keyword article,3515.0,36.5,0.09,12581,down,1
3,content_331d6c4de07b,keyword article,NaN,6.2,0.49,11751,stable,0
4,content_d99b7a2d90ca,keyword article,2803.0,44.0,0.13,19140,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The reference pipeline already answers this empirically: a hand-written rule
(`baseline_rules`, e.g. "flag pages with falling position and low freshness") scores
**precision@50 = 0.24** — barely better than guessing on a population that's 54% declining.
A random forest trained on the same features reaches **0.74**, a ~3x improvement, with no
new data added — just a model that can weigh interacting signals a fixed rule can't.

The top feature importances from that model (`days_with_impressions`, `log_impressions_90d`,
`avg_position`, `content_age_days`, `ctr`, `scroll_rate`...) show *why*: no single one of
these dominates. Decline risk comes from combinations — e.g. an old page with falling
position but still-decent CTR behaves differently than a fresh page with the same falling
position. A rule can check two or three conditions at once before it becomes unreadable;
a model can weigh all ten-plus signals simultaneously, which is exactly the gap this lane
needs closed.

In [ ]:
# Framing only — the supporting evidence is the outputs/model_report.md comparison table above.
print("Baseline rule precision@50: 0.240")
print("Best model (random_forest) precision@50: 0.740")
print(f"Improvement: {0.740/0.240:.2f}x")

Baseline rule precision@50: 0.240
Best model (random_forest) precision@50: 0.740
Improvement: 3.08x


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.